# Chain-of-Thought 복습과제 — 중간 추론 단계가 왜 도움이 될까?

**논문:** Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (NeurIPS 2022)

이 과제에서는 PaLM 540B나 GPT-3 175B를 재현하지 않습니다. 대신 작은 공개 causal language model과 간단한 문자열 처리로 **standard prompting과 chain-of-thought prompting의 차이**를 확인하고, 논문의 주요 결과와 ablation 설계를 복습합니다.

| Part | 내용 | 논문 연결 |
|---|---|---|
| 1 | Standard / Chain-of-Thought prompt 구성 | §2, Figure 1 |
| 2 | 답 추출 + 작은 LM으로 CoT 관찰 | §3.1, §3.2 |
| 3 | 모델 규모와 문제 난이도에 따른 결과 해석 | §3.2, Table 2–3 |
| 4 | Ablation prompt 직접 만들기 | §3.3, Figure 5, Table 6 |
| 5 | Symbolic reasoning과 길이 일반화 | §5, Table 5 |
| 6 | 서술형 복습 | §6–8 |

> 목표는 코드를 어렵게 구현하는 것이 아니라, **prompt에서 정확히 무엇이 바뀌었고, 어떤 가설을 배제하려는 실험인지** 이해하는 것입니다.

In [1]:
# 0. 환경 설정
!pip -q install transformers

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


device: cpu


---
## Part 1. Standard prompt / Chain-of-Thought prompt 만들기

논문에서 두 방식의 차이는 **few-shot exemplar의 답 부분에 chain of thought(중간 추론 단계)를 포함하느냐**입니다.

- Standard prompting: `<input, output>` 쌍
- Chain-of-thought prompting: `<input, chain of thought, output>` triple

두 경우 모두 model weight는 업데이트하지 않습니다. 아래 exemplar는 논문 Appendix Table 20의 수학 문장제 prompt에서 가져왔습니다.

### 문제 1-1. `build_prompt`의 빈칸을 채우세요

- `cot=False`: `A: The answer is {answer}.`
- `cot=True`: `A: {chain} The answer is {answer}.`

In [2]:
exemplars = [
    {
        "question": "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?",
        "chain": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6.",
        "equation": "21 - 15 = 6",
        "answer": "6",
    },
    {
        "question": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
        "chain": "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5.",
        "equation": "3 + 2 = 5",
        "answer": "5",
    },
    {
        "question": "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        "chain": "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39.",
        "equation": "32 + 42 - 35 = 39",
        "answer": "39",
    },
    {
        "question": "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
        "chain": "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8.",
        "equation": "23 - 5 * 3 = 8",
        "answer": "8",
    },
]

def format_answer(ex, cot):
    if cot:
        return f"{ex['chain']} The answer is {ex['answer']}."   # TODO ① chain of thought + 최종 답
    return f"The answer is {ex['answer']}."

def build_prompt(exemplars, query, cot=False):
    prompt = ""
    for ex in exemplars:
        prompt += f"Q: {ex['question']}\nA: {format_answer(ex, cot)}\n\n"
    prompt += f"Q: {query}\nA:"
    return prompt

query = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"

print("===== Standard =====")
print(build_prompt(exemplars[:2], query, cot=False))
print("\n===== Chain-of-Thought =====")
print(build_prompt(exemplars[:2], query, cot=True))

===== Standard =====
Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: The answer is 5.

Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A:

===== Chain-of-Thought =====
Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: There are originally 3 cars. 2 more cars arrive. 3

In [3]:
# ✅ 확인 셀
std = build_prompt(exemplars, query, cot=False)
cot = build_prompt(exemplars, query, cot=True)
assert "21 - 15 = 6" not in std
assert "21 - 15 = 6. The answer is 6." in cot
assert std.endswith(f"Q: {query}\nA:") and cot.endswith(f"Q: {query}\nA:")
print("✅ prompt 구성 확인")

✅ prompt 구성 확인


### 문제 1-2 (서술형)
Standard prompting → Chain-of-thought prompting으로 바꿀 때 **변하는 것**과 **변하지 않는 것**을 각각 쓰세요. exemplar 개수, test query의 형태, model parameter의 update 여부를 포함해 설명하세요.

**답:**

- **변하는 것:** few-shot exemplar의 **답(A) 부분**만 바뀝니다. Standard는 `<input, output>` 쌍이라 `The answer is 6.`처럼 최종 답만 보여 주고, CoT는 `<input, chain of thought, output>` triple이라 `There are 15 trees originally. ... 21 - 15 = 6. The answer is 6.`처럼 자연어로 된 중간 추론 단계를 답 앞에 붙입니다. 그 결과 모델도 답 전에 추론 과정을 먼저 생성하게 되고, 출력 길이(생성 토큰 수)가 늘어납니다.
- **변하지 않는 것:**
  - **exemplar 개수와 질문**: 같은 exemplar 질문을 같은 개수만큼 씁니다 (논문은 수학 문장제에 8-shot. 위 코드에서는 두 prompt 모두 4개).
  - **test query의 형태**: 두 방식 모두 `Q: {query}\nA:`로 끝나고, test 질문에는 추론 과정을 넣지 않습니다. 확인 셀에서 두 prompt의 끝이 똑같은 것으로 확인했습니다.
  - **model parameter**: 두 방식 모두 gradient update나 fine-tuning 없이, 고정된 사전학습 모델 하나에 prompt만 바꿔 넣는 in-context few-shot 방식입니다. decoding도 똑같이 greedy입니다.

---
## Part 2. 답 추출과 작은 Language Model로 CoT 관찰하기

CoT prompting에서는 모델이 긴 추론 문장을 생성하므로, 채점하려면 생성 결과에서 **최종 답만 추출**해야 합니다. 논문의 exemplar는 모두 `The answer is X.`로 끝나므로, 모델도 같은 형식을 따르도록 유도됩니다.

### 문제 2-1. 생성 결과에서 최종 답을 추출하세요

`"The answer is"` 뒤에 나오는 첫 번째 정수를 문자열로 반환하고, 없으면 `None`을 반환합니다.

In [4]:
def extract_answer(text):
    match = re.search(r"The answer is\s*(-?\d+)", text)   # TODO ② "The answer is" 뒤의 (음수 포함) 정수를 찾는 정규식
    return match.group(1) if match else None

print(extract_answer("Roger started with 5 balls. 2 cans of 3 balls is 6. 5 + 6 = 11. The answer is 11."))
print(extract_answer("I am not sure."))

11
None


In [5]:
# ✅ 확인 셀
assert extract_answer("5 + 6 = 11. The answer is 11.") == "11"
assert extract_answer("The answer is -3. Q: next") == "-3"
assert extract_answer("There are 15 trees. 21 - 15 = 6.") is None
print("✅ 답 추출 확인")

✅ 답 추출 확인


### 문제 2-2. 같은 query를 standard / CoT prompt로 비교하세요

실제 논문의 모델은 공개되어 있지 않거나 너무 큽니다. 여기서는 가벼운 `distilgpt2`(약 82M parameter)를 사용해 **prompt만 바꿨을 때 출력이 어떻게 달라지는지** 관찰합니다.

> 작은 모델이 CoT로 문제를 풀지 못하는 것은 정상입니다. 오히려 이것이 논문의 중요한 관찰과 연결됩니다.

In [6]:
MODEL_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
tokenizer.pad_token = tokenizer.eos_token

@torch.no_grad()
def generate(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                         # 논문과 같이 greedy decoding
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    completion = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return completion.split("\nQ:")[0].strip()   # 다음 문제를 생성하기 시작하면 자르기

print("model loaded:", MODEL_ID)

W1123 22:39:13.650000 68019 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

model loaded: distilgpt2


In [7]:
test_set = [
    (query, "11"),
    ("A baker made 12 cookies and ate 4 of them. Then she made 6 more. How many cookies does she have now?", "14"),
]

for q, gold in test_set:
    for use_cot in [False, True]:
        prompt = build_prompt(exemplars, q, cot=use_cot)   # TODO ③ build_prompt로 4-shot prompt 생성 (use_cot 사용)
        completion = generate(prompt)
        pred = extract_answer(completion)
        tag = "CoT     " if use_cot else "Standard"
        print(f"[{tag}] pred={pred} gold={gold} | {completion[:120]}")
    print()

[Standard] pred=8 gold=11 | The answer is 8.


[CoT     ] pred=None gold=11 | Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls is he 



[Standard] pred=9 gold=14 | The answer is 9.


[CoT     ] pred=8 gold=14 | She has 23 cookies. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 



### 문제 2-3 (서술형)
1. 위 실험에서 `distilgpt2`가 생성한 chain of thought는 어떤 특징을 보이나요? (형식은 따라 하는지, 계산은 맞는지)
2. 이 결과만 보고 “CoT prompting은 효과가 없다”고 결론 내릴 수 없는 이유를 논문의 Table 2(예: LaMDA 420M, GPT 350M의 결과)와 연결해 쓰세요.

**답:**

1. **형식은 일부 흉내 내지만 내용과 계산은 맞지 않습니다.**
   - Standard prompt에서는 `The answer is 8.`, `The answer is 9.`처럼 출력 *형식*은 정확히 따랐지만, 숫자는 문제와 상관없이 정답(11, 14)과 다릅니다.
   - CoT prompt에서 첫 번째 문제(Roger)는 추론 없이 **질문 문장을 그대로 반복**하다가 `The answer is`에 이르지 못해 답이 추출되지 않았습니다 (`pred=None`).
   - 두 번째 문제(쿠키)는 문제를 푸는 대신 **4번째 exemplar(Olivia, bagel)의 chain을 거의 그대로 복사**했습니다 ("She has 23 cookies. 5 bagels for 3 dollars each ... 23 - 15 ..."). 문장 구조와 수식 모양은 그럴듯하지만, 숫자도 대상도 질문과 전혀 맞지 않습니다. 결국 문제를 이해하지 못한 채 앞의 문맥을 표면적으로 복사하는 수준입니다.
2. **distilgpt2(82M)는 CoT가 효과를 내기에는 너무 작은 모델입니다.** 논문 Table 2를 보면 작은 모델에서는 CoT가 오히려 성능을 떨어뜨립니다: LaMDA 420M 2.6% → 0.4%, GPT 350M 2.2% → 0.5%. 반대로 같은 방법이 GPT-3 175B에서는 15.6% → 46.9%, PaLM 540B에서는 17.9% → 56.9%로 크게 올라갑니다. 논문의 주장은 CoT가 **충분히 큰 모델(약 100B 이상)에서만 나타나는 emergent ability**라는 것입니다. 따라서 82M 모델에서 효과가 없다는 결과는 오히려 논문의 관찰과 일치하며, CoT prompting 자체가 효과가 없다는 증거가 되지 않습니다. 또 test 문제가 2개뿐이라 정확도를 비교할 만한 표본도 아닙니다.

---
## Part 3. 논문의 주요 결과 읽기

아래는 논문 Table 2(GSM8K)와 Table 3(MAWPS subset)의 일부입니다. 숫자를 직접 계산하며 두 가지 주장을 확인합니다.

- **주장 1:** CoT는 모델 규모에 따른 *emergent ability*이다.
- **주장 2:** 문제가 복잡할수록 CoT의 이득이 크다.

### 문제 3-1. 모델 크기별 CoT 이득을 계산하세요

In [8]:
# GSM8K accuracy (%) — Table 2
gsm8k = {
    "LaMDA": {"420M": (2.6, 0.4), "2B": (3.6, 1.9), "8B": (3.2, 1.6), "68B": (5.7, 8.2), "137B": (6.5, 14.3)},
    "GPT":   {"350M": (2.2, 0.5), "1.3B": (2.4, 0.5), "6.7B": (4.0, 2.4), "175B": (15.6, 46.9)},
    "PaLM":  {"8B": (4.9, 4.1), "62B": (9.6, 29.9), "540B": (17.9, 56.9)},
}   # (standard, CoT)

def cot_gain(standard, cot):
    return cot - standard                        # TODO ④ CoT - standard

for family, sizes in gsm8k.items():
    row = []
    for size, (s, c) in sizes.items():
        row.append(f"{size}: {cot_gain(s, c):+.1f}")
    print(f"{family:6s} | " + "  ".join(row))

LaMDA  | 420M: -2.2  2B: -1.7  8B: -1.6  68B: +2.5  137B: +7.8
GPT    | 350M: -1.7  1.3B: -1.9  6.7B: -1.6  175B: +31.3
PaLM   | 8B: -0.8  62B: +20.3  540B: +39.0


In [9]:
# ✅ 확인 셀
assert round(cot_gain(*gsm8k["PaLM"]["540B"]), 1) == 39.0
assert round(cot_gain(*gsm8k["GPT"]["350M"]), 1) == -1.7
print("✅ 계산 확인")

✅ 계산 확인


### 문제 3-2. 문제 난이도에 따른 이득을 비교하세요

MAWPS의 SingleOp은 한 단계 연산 문제, MultiArith는 여러 단계 연산이 필요한 문제입니다.

In [10]:
# PaLM 540B, MAWPS subsets — Table 3  (standard, CoT)
mawps_palm540 = {
    "SingleOp":   (94.1, 94.1),
    "SingleEq":   (86.5, 92.3),
    "AddSub":     (93.9, 91.9),
    "MultiArith": (42.2, 94.7),
}

for subset, (s, c) in mawps_palm540.items():
    print(f"{subset:10s} | standard {s:5.1f} → CoT {c:5.1f} | gain {cot_gain(s, c):+.1f}")

SingleOp   | standard  94.1 → CoT  94.1 | gain +0.0
SingleEq   | standard  86.5 → CoT  92.3 | gain +5.8
AddSub     | standard  93.9 → CoT  91.9 | gain -2.0
MultiArith | standard  42.2 → CoT  94.7 | gain +52.5


### 문제 3-3 (서술형)
1. 세 모델 family에서 CoT 이득이 **양수로 바뀌는 지점**은 대략 어느 규모인가요? 작은 모델에서는 왜 오히려 성능이 떨어질 수 있을까요? (논문 §3.2의 설명을 참고하세요.)
2. MAWPS 결과에서 SingleOp과 MultiArith의 이득 차이는 무엇을 보여주나요?

**답:**

1. **이득이 양수로 바뀌는 지점:** 위 계산 결과에서
   - LaMDA는 8B(−1.6) → **68B(+2.5)**, 137B에서 +7.8
   - GPT는 6.7B(−1.6) → **175B(+31.3)**
   - PaLM은 8B(−0.8) → **62B(+20.3)**, 540B에서 +39.0

   대략 **수십 B~100B 규모를 넘어야** 이득이 양수가 되고, 의미 있게 커지는 것은 100B 이상입니다. 그보다 작은 모델은 세 family 모두 이득이 음수입니다. 논문 §3.2에 따르면 작은 모델은 **유창하지만 비논리적인 chain of thought**를 만듭니다. 형식은 따라 하지만 계산 실수, 숫자를 잘못 옮겨 적기, 앞뒤가 맞지 않는 단계를 만들고, 추론이 길어질수록 오류가 쌓여 최종 답이 틀리게 됩니다. Standard처럼 바로 답을 찍는 것보다 오히려 나빠질 수 있는 이유입니다. Part 2의 distilgpt2가 exemplar를 복사한 현상도 같은 맥락입니다.
2. **MAWPS 결과:** 한 단계 연산인 **SingleOp은 이득이 0(94.1 → 94.1)**이고, SingleEq(+5.8)와 AddSub(−2.0)도 변화가 작습니다. 반면 여러 단계 연산이 필요한 **MultiArith는 42.2 → 94.7로 +52.5**나 오릅니다. 즉 CoT의 이득은 **문제가 여러 단계의 추론을 요구할수록 커집니다.** 한 단계 문제는 standard prompting만으로도 이미 잘 풀리기 때문에 개선할 여지가 거의 없습니다. 문제를 중간 단계로 쪼개는 CoT는 multi-step 문제에서 진가를 발휘합니다 (논문의 주장 2).

---
## Part 4. Ablation prompt 직접 만들기

CoT가 잘 되는 이유에 대해 다른 설명도 가능합니다. 논문은 아래 세 가지 변형 prompt로 각각의 대안 가설을 검사했습니다 (§3.3, Figure 5).

| Ablation | exemplar의 답 형태 | 검사하려는 가설 |
|---|---|---|
| Equation only | `A: 21 - 15 = 6. The answer is 6.` | 수식만 있으면 충분한가? |
| Variable compute only | `A: ........... The answer is 6.` | 토큰(연산량)이 늘어난 덕분인가? |
| Reasoning after answer | `A: The answer is 6. There are 15 trees ...` | 답을 낸 뒤의 설명도 같은 효과인가? |

### 문제 4-1. 세 가지 ablation 형식을 구현하세요

- Variable compute only: 논문은 **수식에 필요한 문자 수만큼의 점(`.`)**을 사용했습니다. 여기서는 `equation` 문자열 길이만큼 점을 찍습니다.

In [11]:
def format_ablation(ex, mode):
    if mode == "equation_only":
        return f"{ex['equation']}. The answer is {ex['answer']}."
    if mode == "variable_compute":
        dots = "." * len(ex["equation"])          # TODO ⑤ equation 길이만큼의 "."
        return f"{dots} The answer is {ex['answer']}."
    if mode == "reasoning_after_answer":
        return f"The answer is {ex['answer']}. {ex['chain']}"   # TODO ⑥ 답을 먼저, chain of thought를 뒤에
    raise ValueError(mode)

def build_ablation_prompt(exemplars, query, mode):
    prompt = ""
    for ex in exemplars:
        prompt += f"Q: {ex['question']}\nA: {format_ablation(ex, mode)}\n\n"
    prompt += f"Q: {query}\nA:"
    return prompt

for mode in ["equation_only", "variable_compute", "reasoning_after_answer"]:
    print(f"--- {mode} ---")
    print(format_ablation(exemplars[0], mode), "\n")

--- equation_only ---
21 - 15 = 6. The answer is 6. 

--- variable_compute ---
........... The answer is 6. 

--- reasoning_after_answer ---
The answer is 6. There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. 



In [12]:
# ✅ 확인 셀
ex0 = exemplars[0]
assert format_ablation(ex0, "variable_compute") == "." * len(ex0["equation"]) + " The answer is 6."
after = format_ablation(ex0, "reasoning_after_answer")
assert after.startswith("The answer is 6.") and ex0["chain"] in after
assert after.index("The answer is") < after.index(ex0["chain"])
print("✅ ablation prompt 확인")

✅ ablation prompt 확인


### 문제 4-2. 논문의 ablation 결과를 정리하세요

In [13]:
# LaMDA 137B, GSM8K accuracy (%) — Table 6
ablation_gsm8k = {
    "standard": 6.5,
    "equation_only": 5.4,
    "variable_compute": 6.4,
    "reasoning_after_answer": 6.1,
    "chain_of_thought": 14.3,
}

best_ablation = max(
    (k for k in ablation_gsm8k if k not in ("standard", "chain_of_thought")),
    key=ablation_gsm8k.get,
)                                                # TODO ⑦ standard, chain_of_thought를 제외한 ablation 중 가장 높은 것의 이름
print("best ablation:", best_ablation, ablation_gsm8k[best_ablation])
print("CoT - best ablation:", round(ablation_gsm8k["chain_of_thought"] - ablation_gsm8k[best_ablation], 1))

best ablation: variable_compute 6.4
CoT - best ablation: 7.9


In [14]:
# ✅ 확인 셀
assert best_ablation == "variable_compute"
print("✅ 결과 확인")

✅ 결과 확인


### 문제 4-3 (서술형)
1. 세 ablation은 각각 어떤 대안 가설을 **배제**하나요? 위 결과를 근거로 한 문장씩 쓰세요.
2. Equation only는 GSM8K에서는 거의 도움이 되지 않았지만, 한두 단계로 풀리는 쉬운 데이터셋(SVAMP, ASDiv 등)에서는 성능을 올렸습니다 (Table 6). 이 차이는 무엇을 의미할까요?

**답:**

1. LaMDA 137B, GSM8K 기준 (standard 6.5%, CoT 14.3%):
   - **Equation only (5.4%)**: CoT의 효과가 단순히 *문제를 수식으로 바꿔 주기 때문*이라는 가설을 배제합니다. GSM8K처럼 의미가 복잡한 문제는 자연어 추론 없이 곧바로 수식으로 옮기기 어려워, 수식만으로는 standard보다도 나아지지 않았습니다.
   - **Variable compute only (6.4%)**: CoT의 효과가 *답 전에 토큰을 더 생성해 연산량이 늘어났기 때문*이라는 가설을 배제합니다. 수식 길이만큼 점을 찍어 연산량을 맞춰 줘도 standard와 거의 같았으므로, 중요한 것은 토큰의 개수가 아니라 **중간 단계를 자연어로 표현한 내용**입니다.
   - **Reasoning after answer (6.1%)**: CoT의 효과가 *추론 과정을 보여 주는 exemplar가 사전학습 지식을 더 잘 불러오기 때문*이라는 가설을 배제합니다. 같은 chain을 답 *뒤에* 두면 효과가 사라지므로, 추론이 **답을 생성하기 전에 순서대로 나와서** 답이 그 추론에 의존해야 합니다.

   세 ablation 중 가장 높은 variable compute(6.4%)도 CoT(14.3%)보다 7.9%p 낮습니다.
2. 쉬운 데이터셋(SVAMP, ASDiv 등)은 한두 단계로 풀리기 때문에 **질문에서 수식이 바로 도출됩니다.** 그래서 수식만 생성하게 해도 도움이 됩니다. 반면 GSM8K는 여러 단계를 거쳐야 하고, 질문의 의미를 해석하는 과정 없이 한 번에 정확한 수식으로 옮기기 어렵습니다. 결국 **수식은 최종 계산을 표현할 뿐이고, 어려운 문제에서는 의미를 해석하고 단계를 나누는 *자연어* 중간 추론이 핵심**이라는 뜻입니다. 이는 문제가 복잡할수록 CoT의 이득이 크다는 Part 3의 결과와도 일치합니다.

---
## Part 5. Symbolic Reasoning과 길이 일반화

Last letter concatenation은 이름에 포함된 각 단어의 마지막 글자를 이어 붙이는 task입니다.
예: `"Elon Musk"` → `"nk"`

논문은 exemplar를 **2단어 이름**으로만 구성하고, test에서 **3·4단어 이름(OOD)**을 사용해 길이 일반화를 확인했습니다.

### 문제 5-1. 정답 함수와 chain of thought 생성 함수를 완성하세요

`make_chain`은 논문 Table 22의 exemplar 형식을 그대로 따릅니다.

In [15]:
def last_letter_concat(name):
    return "".join(w[-1] for w in name.split())  # TODO ⑧ 각 단어의 마지막 글자를 이어 붙이기

def make_chain(name):
    words = name.split()
    steps = " ".join(f'The last letter of "{w}" is "{w[-1]}".' for w in words)
    ans = last_letter_concat(name)
    return f'{steps} Concatenating them is "{ans}". The answer is {ans}.'

def letter_prompt(name):
    return f'Q: Take the last letters of the words in "{name}" and concatenate them.\nA:'

print(make_chain("Elon Musk"))
print(make_chain("Bill Henry Gates"))   # OOD: 3단어

The last letter of "Elon" is "n". The last letter of "Musk" is "k". Concatenating them is "nk". The answer is nk.
The last letter of "Bill" is "l". The last letter of "Henry" is "y". The last letter of "Gates" is "s". Concatenating them is "lys". The answer is lys.


In [16]:
# ✅ 확인 셀
assert last_letter_concat("Elon Musk") == "nk"
assert last_letter_concat("Larry Page") == "ye"
assert last_letter_concat("Bill Henry Gates") == "lys"
assert make_chain("Sergey Brin").endswith('Concatenating them is "yn". The answer is yn.')
print("✅ symbolic task 확인")

✅ symbolic task 확인


In [17]:
# Last letter concatenation accuracy (%) — Table 5, PaLM  (standard, CoT)
letter_concat = {
    "8B":   {"in-domain (2)": (2.6, 18.8), "OOD (3)": (0.0, 0.0),  "OOD (4)": (0.0, 0.2)},
    "62B":  {"in-domain (2)": (6.8, 85.0), "OOD (3)": (0.0, 59.6), "OOD (4)": (0.0, 13.4)},
    "540B": {"in-domain (2)": (7.6, 99.4), "OOD (3)": (0.2, 94.8), "OOD (4)": (0.0, 63.0)},
}

for size, res in letter_concat.items():
    print(f"PaLM {size:5s} | " + " | ".join(f"{k}: {s:4.1f} → {c:4.1f}" for k, (s, c) in res.items()))

PaLM 8B    | in-domain (2):  2.6 → 18.8 | OOD (3):  0.0 →  0.0 | OOD (4):  0.0 →  0.2
PaLM 62B   | in-domain (2):  6.8 → 85.0 | OOD (3):  0.0 → 59.6 | OOD (4):  0.0 → 13.4
PaLM 540B  | in-domain (2):  7.6 → 99.4 | OOD (3):  0.2 → 94.8 | OOD (4):  0.0 → 63.0


### 문제 5-2 (서술형)
1. Standard prompting은 OOD에서 거의 0%인데 CoT는 PaLM 540B에서 94.8%(3단어)를 기록합니다. CoT의 어떤 구조가 **exemplar보다 긴 입력**으로의 일반화를 가능하게 할까요? 위의 `make_chain` 출력 형태를 참고하세요.
2. 이 task는 정답을 구하는 절차가 exemplar에 그대로 드러나 있는 “toy task”입니다. 그런데도 PaLM 8B는 in-domain에서 18.8%에 그칩니다. 이 결과는 무엇을 시사하나요?

**답:**

1. `make_chain` 출력은 `The last letter of "Bill" is "l". The last letter of "Henry" is "y". ...`처럼 **단어마다 같은 형식의 단계를 하나씩 반복**한 뒤 `Concatenating them is "..."`로 합칩니다. 즉 CoT는 task를 **단어 단위의 단순한 sub-step으로 분해한 재귀적 절차**를 보여 줍니다. 모델이 이 *단위 절차*를 배우면, 입력 단어가 3·4개로 늘어나도 같은 단계를 한 번 더 반복하기만 하면 됩니다. 각 단계의 난이도는 그대로이고 chain의 길이만 늘어납니다. 반면 standard prompting은 이름 전체에서 정답 문자열로 한 번에 넘어가는 매핑만 보여 주므로, exemplar에서 본 적 없는 길이에는 거의 일반화하지 못합니다 (≈0%). 다만 단계가 길어질수록 오류가 쌓일 가능성도 커져서, 540B에서도 OOD(4)는 63.0%로 OOD(3)의 94.8%보다 낮습니다.
2. 풀이 절차가 exemplar에 완전히 드러나 있는데도 8B 모델은 in-domain 18.8%, OOD는 거의 0%에 그칩니다. 이는 **CoT의 효과가 "절차를 보여 주는 것"만으로 생기지 않고, 그 절차를 새 입력에 정확히 적용할 만큼의 모델 규모가 필요하다**는 뜻입니다. 단어 분리, 마지막 글자 찾기, 이어 붙이기 같은 기본 조작을 안정적으로 하고 형식을 일관되게 유지하는 능력이 작은 모델에는 부족합니다. 62B(85.0%), 540B(99.4%)로 커져야 이 능력이 나타나므로, symbolic reasoning에서도 CoT는 **모델 규모에 따른 emergent ability**입니다.

---
## Part 6. 서술형 복습

**6-1.** CoT prompting과 Ling et al. (2017), Cobbe et al. (2021)처럼 rationale을 생성하도록 모델을 학습/파인튜닝하는 방법의 가장 중요한 차이는 무엇인가요? 각 방법의 비용 측면을 포함해 쓰세요.

**답:**

가장 큰 차이는 **parameter를 학습하느냐**입니다.
- Ling et al. (2017)과 Cobbe et al. (2021)은 사람이 작성한 **대규모 rationale 데이터셋**을 만들어 모델을 처음부터 학습하거나 fine-tuning합니다. 따라서 (1) 고품질 중간 추론을 대량으로 annotation하는 **데이터 구축 비용**과 (2) 큰 모델을 학습시키는 **연산 비용**이 들고, (3) task마다 별도의 학습과 별도의 모델 checkpoint가 필요합니다.
- CoT prompting은 이미 사전학습된 대형 모델을 **그대로 두고**, 손으로 쓴 몇 개(약 8개)의 exemplar만 prompt에 넣습니다. annotation 비용이 거의 없고 학습 비용도 0입니다. 같은 모델 하나로 prompt만 바꿔 수학, 상식, symbolic 등 여러 task를 다룰 수 있어 범용성이 높습니다.
- 대신 CoT prompting은 그 비용을 **추론 시점과 모델 규모**로 넘깁니다. 100B급 대형 모델이 있어야 효과가 나타나고, 매번 긴 prompt와 긴 출력을 처리해야 합니다. 반면 fine-tuning 방식은 상대적으로 작은 모델로도 해당 task에서 높은 성능을 낼 수 있습니다.

**6-2.** 논문은 “chain of thought가 사람의 사고 과정을 모방하지만, 신경망이 실제로 *reasoning*을 하는지는 열린 질문”이라고 말합니다. 정답을 맞힌 CoT 중에도 추론 과정이 틀린 경우가 있다는 점(Appendix D.1)은 CoT의 해석 가능성(interpretability) 주장에 어떤 한계를 주나요?

**답:**

CoT의 장점 중 하나는 모델이 *어떻게* 답에 도달했는지 사람이 읽을 수 있다는 해석 가능성입니다. 그런데 Appendix D.1의 분석처럼 **정답을 맞힌 chain 중에도 추론이 틀린 경우가 있다면** (예: 중간 계산이 틀렸는데 우연히 최종 답이 맞거나, 논리가 맞지 않는 단계가 섞인 경우), 생성된 chain은 **모델이 실제로 답을 계산한 과정을 충실히(faithfully) 보여 준다는 보장이 없습니다.** chain은 사람이 읽기에 그럴듯한 *사후 설명*일 수도 있고, 모델 내부의 실제 계산과 다를 수 있습니다. 따라서 (1) chain이 맞아 보인다고 답을 신뢰하거나, (2) 정답이니 추론도 맞았다고 판단하는 것은 위험하고, (3) chain을 보고 모델의 오류 원인을 디버깅하는 데에도 한계가 있습니다. CoT는 디버깅 *단서*가 될 수는 있지만 신경망 내부 추론에 대한 직접적인 설명은 아니며, 실제 reasoning을 하는지는 여전히 열린 질문입니다.

**6-3.** 논문은 “standard prompting은 대형 언어 모델 능력의 *lower bound*일 뿐”이라고 말합니다. 이 주장의 의미를 Part 3의 결과와 연결해 설명하세요.

**답:**

Standard prompting의 성능은 모델이 가진 능력 중 *그 prompt 방식으로 끌어낼 수 있는 부분*만 보여 주므로, 모델의 실제 능력보다 **낮게 측정될 수 있다**는 뜻입니다. Part 3에서 같은 PaLM 540B가 GSM8K에서 standard로는 17.9%였지만 prompt만 CoT로 바꾸면 56.9%가 됩니다 (+39.0%p, GPT-3 175B도 15.6% → 46.9%). MultiArith에서도 42.2% → 94.7%로 뜁니다. model weight는 전혀 바뀌지 않았으므로, 이 능력은 원래 모델 안에 있었는데 standard prompting이 끌어내지 못했던 것입니다. 또 standard prompting에서는 모델 크기를 키워도 성능 곡선이 비교적 평평해 multi-step 추론은 규모로 해결되지 않는 것처럼 보였지만, CoT로는 규모에 따라 성능이 가파르게 오릅니다. 따라서 standard prompting 결과만 보고 대형 언어 모델은 추론을 못 한다고 결론 내리면 안 되며, 더 좋은 prompting 방법이 능력의 상한을 계속 끌어올릴 수 있습니다.

**6-4.** CoT prompting은 정확도를 높이지만, 모델이 생성하는 토큰 수를 크게 늘립니다. 실제 서비스 관점에서 이것이 어떤 비용(latency, 메모리 등)을 만드는지 쓰고, 논문의 Limitations에서 언급한 “큰 모델에서만 효과가 나타난다”는 점과 함께 실용성을 평가하세요.

**답:**

- **비용:** CoT는 답 하나를 내기 위해 수십~수백 개의 추론 토큰을 더 생성합니다. autoregressive decoding은 토큰을 하나씩 순서대로 만들기 때문에 **latency가 출력 길이에 비례해 늘어납니다.** 생성 길이가 길어지면 **KV cache 메모리**도 커져서 GPU 한 대가 동시에 처리할 수 있는 요청 수(throughput)가 줄어듭니다. 게다가 prompt에 긴 CoT exemplar(8-shot)가 매번 들어가므로 **입력 토큰 수**도 늘어납니다. API 기준으로는 입력과 출력 토큰 모두에 과금되므로 요청당 비용이 몇 배로 커집니다.
- **규모 제약과 합친 실용성 평가:** Limitations에서 말하듯 CoT의 효과는 **약 100B 이상의 대형 모델에서만** 나타나고, 작은 모델에서는 오히려 성능이 떨어집니다. 그래서 CoT를 쓰려면 원래도 serving 비용이 큰 대형 모델에 긴 출력까지 추가해야 하므로, **비싼 모델 × 긴 생성이라는 이중 비용**이 생깁니다. 따라서 실시간 응답이 중요하거나 대량 요청을 처리하는 서비스, 혹은 한 단계로 풀리는 쉬운 질의(SingleOp처럼 이득이 0인 경우)에는 비용 대비 효과가 낮습니다. 반대로 multi-step 수학, 복잡한 추론처럼 정확도가 중요하고 standard 대비 이득이 큰 task에서는 추가 비용을 감수할 가치가 있습니다. 결국 **task 난이도에 따라 CoT를 선택적으로 적용**하는 것이 현실적입니다. 또 Limitations에서 언급하듯 chain이 맞다는 보장이 없으므로, 높은 신뢰성이 필요한 서비스에서는 추가 검증 장치도 필요합니다.

---
## 참고 자료
- Wei et al. (2022), *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*: CoT Prompting (§2), Arithmetic Reasoning (§3), Commonsense Reasoning (§4), Symbolic Reasoning (§5), Discussion (§6)
- Appendix B (Table 1–7): 모델·규모별 전체 결과, Appendix G: 실험에 사용한 전체 few-shot prompt

> 이 과제는 대형 모델의 원 실험을 재현하는 과제가 아니라, CoT prompting의 핵심 아이디어와 실험 설계를 작은 예제로 복습하기 위한 과제입니다.